# Système RAG final — Notebook Kaggle

Ce notebook exécute les **six étapes** du projet avec les améliorations : nettoyage réel, traçabilité des sources, embeddings multilingues, recherche hybride, reranking, réponses sourcées, évaluation officielle Ragas et sauvegarde des statistiques pour le rapport de stage.

## Les deux seuls modes

| Mode | Utilisation |
|---|---|
| `PREPARATION` | Premier lancement. Il construit le système amélioré, prépare les 120 questions et les passages candidats à vérifier. Aucun Secret Kaggle n'est nécessaire. |
| `EVALUATION_FINALE` | Dernier lancement. Il exécute Ragas sur les 120 questions déjà validées et sauvegarde les résultats finaux. |

> **Pour commencer : ne changez rien et cliquez sur “Run All”.**


In [ ]:
# ===== CELLULE 1 — Configuration simple =====
# Premier lancement : laissez PREPARATION.
MODE = "PREPARATION"  # "PREPARATION" ou "EVALUATION_FINALE"

# L'interface est facultative. Elle peut être activée après le premier lancement.
LANCER_INTERFACE = False

# Informations nécessaires uniquement pour MODE = "EVALUATION_FINALE".
NOM_SECRET_KAGGLE = "OPENAI_API_KEY"
FOURNISSEUR_JUGE = "openai"
MODELE_JUGE = "gpt-4o-mini"

# Ne modifiez pas ce profil : il correspond au système final amélioré.
PROFIL_RECHERCHE = "multilingual_hybrid_rerank"


## Cellule 2 — Installation et dossier de travail

Cette cellule télécharge la version la plus récente du projet, installe les dépendances et prépare un dossier durable dans `/kaggle/working`.


In [ ]:
from pathlib import Path
import json, os, random, shutil, subprocess, sys
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch

GRAINE = 42
random.seed(GRAINE)
np.random.seed(GRAINE)
torch.manual_seed(GRAINE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GRAINE)

REPO_URL = "https://github.com/DAHANIElkhalil25/rag-pipeline.git"
DOSSIER_PROJET = Path("/kaggle/working/rag-pipeline-final")
DOSSIER_DONNEES = Path("/kaggle/working/rag_final_data")

if DOSSIER_PROJET.exists():
    shutil.rmtree(DOSSIER_PROJET)
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(DOSSIER_PROJET)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=DOSSIER_PROJET, check=True)

os.environ["RAG_DATA_DIR"] = str(DOSSIER_DONNEES)
os.environ["RAG_RETRIEVAL_PROFILE"] = PROFIL_RECHERCHE
sys.path.insert(0, str(DOSSIER_PROJET))
os.chdir(DOSSIER_PROJET)

for nom in ["config", "etape5_generation", "evaluation.ragas_runner"]:
    sys.modules.pop(nom, None)

from config import BENCHMARK_DIR, CLEAN_DIR, EVALUATION_DIR, METADATA_DIR, RAW_DIR, VECTORSTORE_DIR, init_directories
init_directories()

COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print(f"Version du projet : {COMMIT}")
print(f"Matériel : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Mode choisi : {MODE}")


## Cellule 3 — Étapes 1 à 4 : données, nettoyage, benchmarking et index

Cette cellule lance automatiquement les étapes suivantes seulement si les fichiers nécessaires n'existent pas déjà.

1. Collecte de la documentation officielle.
2. Nettoyage et suppression réelle des doublons.
3. Benchmark de récupération.
4. Création de l'index FAISS avec les embeddings multilingues.


In [ ]:
from etape1_collecte import main as etape1_collecte
from etape2_nettoyage import main as etape2_nettoyage
from etape3_benchmarking import main as etape3_benchmarking
from etape4_indexation import main as etape4_indexation

if not (METADATA_DIR / "corpus_index.csv").exists():
    etape1_collecte()
if not (CLEAN_DIR / "corpus_cleaned_index.csv").exists():
    etape2_nettoyage()
if not (BENCHMARK_DIR / "benchmark_report.json").exists():
    etape3_benchmarking()

chemin_manifeste_index = VECTORSTORE_DIR / "index_manifest.json"
reconstruire_index = True
if chemin_manifeste_index.exists():
    ancien = json.loads(chemin_manifeste_index.read_text(encoding="utf-8"))
    ancien_profil = ancien.get("index_config", {}).get("retrieval_profile")
    reconstruire_index = ancien_profil != PROFIL_RECHERCHE
if reconstruire_index:
    etape4_indexation()

manifeste_index = json.loads(chemin_manifeste_index.read_text(encoding="utf-8"))
print("Index final prêt.")
print(json.dumps(manifeste_index.get("index_config", {}), ensure_ascii=False, indent=2))


## Cellule 4 — Étape 5 : système RAG amélioré

Le système récupère 20 passages candidats avec une recherche hybride, les rerank avec un modèle multilingue, conserve les 5 meilleurs passages, puis Mistral génère une réponse accompagnée de sources.


In [ ]:
from etape5_generation import load_pipeline

pipeline = load_pipeline()
question_demo = "Comment éviter une fuite de données lors de la standardisation avant une validation croisée ?"
resultat_demo = pipeline.answer(question_demo)

print("QUESTION :", question_demo)
print("\nRÉPONSE :\n", resultat_demo["answer"])
print("\nSOURCES :")
for rang, chunk in enumerate(resultat_demo.get("retrieved_chunks", []), start=1):
    print(f"[{rang}] {chunk.get('chunk_id')} — {chunk.get('doc_url', chunk.get('doc_source'))}")

# Le reranker n'est plus nécessaire après la récupération des candidats ; on libère sa mémoire avant l'évaluation longue.
pipeline._unload_reranker()


## Cellule 5 — Étape 6 : préparation des 120 questions ou évaluation finale

En mode `PREPARATION`, le notebook crée les 120 questions et propose des passages candidats. Les passages ne sont jamais validés automatiquement : ils sont préparés pour une vérification humaine.

En mode `EVALUATION_FINALE`, le notebook utilise le fichier final déjà validé et appelle Ragas.


In [ ]:
from evaluation.bootstrap_datasets import create_development_set, create_final_annotation_template
from evaluation.build_question_bank import build_records, PYTHON_ITEMS, SKLEARN_ITEMS, LANGCHAIN_ITEMS
from evaluation.dataset_schema import write_jsonl

create_development_set()
create_final_annotation_template()

dossier_datasets = DOSSIER_PROJET / "evaluation" / "datasets"
jeu_brouillon = dossier_datasets / "test_dataset_v1_source_grounded_draft.jsonl"
jeu_final = dossier_datasets / "test_dataset_v1.jsonl"

if not jeu_brouillon.exists():
    lignes = (build_records("python", PYTHON_ITEMS) + build_records("scikit_learn", SKLEARN_ITEMS) + build_records("langchain", LANGCHAIN_ITEMS))
    write_jsonl(jeu_brouillon, lignes)

resultat_ragas = None
if MODE == "PREPARATION":
    from evaluation.create_annotation_candidates import create_candidates
    sortie_candidats = DOSSIER_DONNEES / "evaluation" / "annotation_candidates" / "candidats_120_questions.jsonl"
    create_candidates(pipeline, jeu_brouillon, sortie_candidats, k=10)
    pipeline._unload_reranker()
    print("Préparation terminée.")
    print("Envoyez ensuite le fichier suivant dans le chat pour la revue :")
    print(sortie_candidats)
elif MODE == "EVALUATION_FINALE":
    if not jeu_final.exists():
        raise FileNotFoundError("Le fichier test_dataset_v1.jsonl validé est absent. Exécutez d'abord le mode PREPARATION et faites valider les 120 questions.")
    from kaggle_secrets import UserSecretsClient
    from evaluation.ragas_runner import run_final_evaluation
    cle_api = UserSecretsClient().get_secret(NOM_SECRET_KAGGLE)
    id_run = f"final_120_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
    resultat_ragas = await run_final_evaluation(
        pipeline=pipeline, dataset_path=jeu_final, run_id=id_run,
        provider=FOURNISSEUR_JUGE, judge_model=MODELE_JUGE, api_key=cle_api,
    )
    print(json.dumps(resultat_ragas["summary"], ensure_ascii=False, indent=2))
else:
    raise ValueError("MODE doit être PREPARATION ou EVALUATION_FINALE.")


## Cellule 6 — Statistiques pour le rapport de stage

Cette cellule sauvegarde automatiquement les chiffres utiles pour le rapport : taille du corpus, nettoyage, index, chunks, configuration, temps, couverture des questions, résultats Ragas et erreurs. Elle crée aussi deux graphiques simples.


In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

dossier_stats = DOSSIER_DONNEES / "statistiques_rapport"
dossier_stats.mkdir(parents=True, exist_ok=True)

def compter_fichiers(dossier):
    return len([p for p in dossier.rglob("*") if p.is_file()]) if dossier.exists() else 0

chunks = json.loads((VECTORSTORE_DIR / "chunks_metadata.json").read_text(encoding="utf-8"))
chunks_par_source = Counter(chunk.get("doc_source", "inconnu") for chunk in chunks)
documents_par_source = Counter(chunk.get("document_id", "inconnu").split("_")[0] for chunk in chunks)

statistiques = {
    "date_utc": datetime.now(timezone.utc).isoformat(),
    "commit": COMMIT,
    "mode": MODE,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "profil_recherche": PROFIL_RECHERCHE,
    "fichiers_bruts": compter_fichiers(RAW_DIR),
    "fichiers_nettoyes": compter_fichiers(CLEAN_DIR),
    "nombre_chunks": len(chunks),
    "chunks_par_source": dict(chunks_par_source),
    "documents_approx_par_source": dict(documents_par_source),
    "index": manifeste_index,
    "jeu_brouillon_120": str(jeu_brouillon),
    "evaluation_ragas": resultat_ragas["summary"] if resultat_ragas else None,
}

(dossier_stats / "statistiques_rapport.json").write_text(json.dumps(statistiques, ensure_ascii=False, indent=2), encoding="utf-8")
pd.DataFrame([
    {"indicateur": "nombre_chunks", "valeur": len(chunks)},
    {"indicateur": "fichiers_bruts", "valeur": statistiques["fichiers_bruts"]},
    {"indicateur": "fichiers_nettoyes", "valeur": statistiques["fichiers_nettoyes"]},
]).to_csv(dossier_stats / "statistiques_rapport.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(chunks_par_source.keys(), chunks_par_source.values(), color=["#3B82F6", "#F59E0B", "#10B981"])
ax.set_title("Nombre de chunks par source documentaire")
ax.set_ylabel("Nombre de chunks")
fig.tight_layout()
fig.savefig(dossier_stats / "chunks_par_source.png", dpi=180)
plt.show()

if resultat_ragas:
    resume = resultat_ragas["summary"].get("metric_summary", {})
    noms = list(resume)
    valeurs = [resume[nom]["mean"] for nom in noms]
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(noms, valeurs, color="#2563EB")
    ax.set_ylim(0, 1)
    ax.set_title("Résultats officiels Ragas — système final")
    ax.set_ylabel("Score moyen")
    plt.xticks(rotation=20, ha="right")
    fig.tight_layout()
    fig.savefig(dossier_stats / "scores_ragas_finaux.png", dpi=180)
    plt.show()

print("Statistiques sauvegardées dans :", dossier_stats)


## Cellule 7 — Interface utilisateur facultative

Cette cellule lance une interface Gradio pour poser vos propres questions au système final. Elle ne modifie pas les résultats de l'évaluation.


In [ ]:
if LANCER_INTERFACE:
    from config import UI_CONFIG
    from ui import launch_ui
    launch_ui(pipeline, share=UI_CONFIG["share"], server_name=UI_CONFIG["server_name"], server_port=UI_CONFIG["server_port"])
else:
    print("Interface désactivée. Mettez LANCER_INTERFACE = True si vous voulez la tester.")


## Cellule 8 — Export final

Téléchargez ensuite l'archive `resultats_rag_stage.zip` depuis l'onglet **Files** de Kaggle. Elle contient les statistiques, manifestes, passages récupérés, résultats Ragas éventuels et fichiers utiles pour le rapport.


In [ ]:
manifeste_execution = {
    "commit": COMMIT,
    "mode": MODE,
    "profil_recherche": PROFIL_RECHERCHE,
    "date_utc": datetime.now(timezone.utc).isoformat(),
    "fichiers": sorted(str(p.relative_to(DOSSIER_DONNEES)) for p in DOSSIER_DONNEES.rglob("*") if p.is_file()),
}
(DOSSIER_DONNEES / "manifeste_execution.json").write_text(json.dumps(manifeste_execution, ensure_ascii=False, indent=2), encoding="utf-8")

dossier_export = Path("/kaggle/working/resultats_rag_stage")
if dossier_export.exists():
    shutil.rmtree(dossier_export)
shutil.copytree(DOSSIER_DONNEES, dossier_export)
shutil.make_archive("/kaggle/working/resultats_rag_stage", "zip", dossier_export)
print("Archive prête : /kaggle/working/resultats_rag_stage.zip")
